In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings

# ====================== IMPORT TKINTER FOR GUI ======================
import tkinter as tk
from tkinter import ttk, messagebox

print("="*60)
print("TITANIC SURVIVAL PREDICTION PROJECT")
print("="*60)

# ====================== 1. LOAD DATA ======================
df = pd.read_csv('train.csv')
print("✓ Loaded Kaggle Titanic dataset")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")

# ====================== 2. BASIC DATA INFO ======================
print(f"\n📊 Basic Information:")
print(f"   Total passengers: {len(df)}")
print(f"   Survived: {df['Survived'].sum()} ({(df['Survived'].sum()/len(df))*100:.1f}%)")
print(f"   Not Survived: {len(df) - df['Survived'].sum()}")

# Check missing values
print(f"\n🔍 Missing Values:")
missing = df.isnull().sum()
for col, val in missing[missing > 0].items():
    print(f"   {col}: {val} missing")

# ====================== 3. DATA PREPROCESSING ======================
print(f"\n⚙️  Data Preprocessing...")

# Fill missing values
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())  # Fill missing fares

# Create new features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Encode categorical variables
le_sex = LabelEncoder()
df['Sex'] = le_sex.fit_transform(df['Sex'])  # male=0, female=1

le_embarked = LabelEncoder()
df['Embarked'] = le_embarked.fit_transform(df['Embarked'])

# Select features
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']
X = df[features]
y = df['Survived']

# ====================== 4. TRAIN-TEST SPLIT ======================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✓ Data split: Train={X_train.shape[0]}, Test={X_test.shape[0]}")

# ====================== 5. TRAIN MODELS ======================
print(f"\n🤖 Training Models...")

# Model 1: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

# ====================== 6. RESULTS ======================
print(f"\n🏆 RESULTS:")
print(f"   Logistic Regression: {acc_lr:.3f} ({acc_lr*100:.1f}%)")
print(f"   Random Forest: {acc_rf:.3f} ({acc_rf*100:.1f}%)")

# Best model
if acc_rf > acc_lr:
    best_model = rf
    best_acc = acc_rf
    best_name = "Random Forest"
else:
    best_model = lr
    best_acc = acc_lr
    best_name = "Logistic Regression"

print(f"\n✓ Best Model: {best_name} with {best_acc*100:.1f}% accuracy")

# ====================== 7. CREATE SIMPLE GUI ======================
print(f"\n🖥️  Creating GUI...")

def create_gui():
    """Create a simple GUI for predictions"""
    
    # Create main window
    window = tk.Tk()
    window.title("Titanic Survival Predictor")
    window.geometry("500x700")
    window.configure(bg='#f0f0f0')
    
    # Title
    title_label = tk.Label(window, text="🚢 TITANIC SURVIVAL PREDICTOR", 
                          font=('Arial', 18, 'bold'), bg='#f0f0f0', fg='#2c3e50')
    title_label.pack(pady=20)
    
    # Info label
    info_text = f"Best Model: {best_name} ({best_acc*100:.1f}% accuracy)"
    info_label = tk.Label(window, text=info_text, font=('Arial', 12), 
                         bg='#f0f0f0', fg='#34495e')
    info_label.pack(pady=10)
    
    # Create frame for inputs (using pack only)
    input_frame = tk.Frame(window, bg='white', relief=tk.RAISED, bd=2)
    input_frame.pack(pady=20, padx=40, fill='both')
    
    tk.Label(input_frame, text="Enter Passenger Details:", font=('Arial', 14, 'bold'), 
            bg='white').pack(pady=10)
    
    # Create input fields using pack (not grid)
    def create_input_row(parent, label_text, default_value, input_type="entry"):
        # Create a frame for each row
        row_frame = tk.Frame(parent, bg='white')
        row_frame.pack(pady=5, fill='x')
        
        # Label
        label = tk.Label(row_frame, text=label_text, bg='white', font=('Arial', 11), width=25, anchor='w')
        label.pack(side='left', padx=10)
        
        # Entry/Combobox
        var = tk.StringVar(value=default_value)
        
        if input_type == "gender":
            combo = ttk.Combobox(row_frame, textvariable=var, values=["male", "female"], 
                                width=15, state="readonly")
            combo.pack(side='left', padx=10)
        elif input_type == "embarked":
            combo = ttk.Combobox(row_frame, textvariable=var, values=["C", "Q", "S"], 
                                width=15, state="readonly")
            combo.pack(side='left', padx=10)
        elif input_type == "pclass":
            combo = ttk.Combobox(row_frame, textvariable=var, values=["1", "2", "3"], 
                                width=15, state="readonly")
            combo.pack(side='left', padx=10)
        elif input_type == "number":
            # For age, siblings, parents - use spinbox for better control
            spinbox = tk.Spinbox(row_frame, from_=0, to=100, textvariable=var, width=13)
            spinbox.pack(side='left', padx=10)
        elif input_type == "fare":
            # Special handling for fare with validation
            def validate_fare(P):
                if P == "":
                    return True
                try:
                    value = float(P)
                    if 0 <= value <= 1000:  # Reasonable fare range for Titanic
                        return True
                    else:
                        return False
                except:
                    return False
            
            vcmd = (window.register(validate_fare), '%P')
            entry = tk.Entry(row_frame, textvariable=var, width=18, 
                           validate="key", validatecommand=vcmd)
            entry.pack(side='left', padx=10)
        else:
            entry = tk.Entry(row_frame, textvariable=var, width=18)
            entry.pack(side='left', padx=10)
        
        return var
    
    # Create input variables with proper validation
    pclass_var = create_input_row(input_frame, "Passenger Class:", "1", "pclass")
    gender_var = create_input_row(input_frame, "Gender:", "female", "gender")
    age_var = create_input_row(input_frame, "Age:", "30", "number")
    sibsp_var = create_input_row(input_frame, "Siblings/Spouses:", "0", "number")
    parch_var = create_input_row(input_frame, "Parents/Children:", "0", "number")
    fare_var = create_input_row(input_frame, "Ticket Fare (£):", "50", "fare")
    embarked_var = create_input_row(input_frame, "Embarked From:", "S", "embarked")
    
    # Result frame
    result_frame = tk.Frame(window, bg='#ecf0f1', relief=tk.GROOVE, bd=2)
    result_frame.pack(pady=20, padx=40, fill='x')
    
    result_label = tk.Label(result_frame, text="Enter details and click PREDICT", 
                           font=('Arial', 14), bg='#ecf0f1', fg='gray', height=3, wraplength=350)
    result_label.pack(pady=20)
    
    def predict_survival():
        try:
            # Get values from input fields
            pclass = int(pclass_var.get())
            gender = 1 if gender_var.get() == "female" else 0
            age = float(age_var.get())
            sibsp = int(sibsp_var.get())
            parch = int(parch_var.get())
            fare = float(fare_var.get())
            
            # Validate fare range
            if fare < 0 or fare > 1000:
                messagebox.showerror("Input Error", "Fare must be between £0 and £1000")
                return
            
            # Convert embarked to number
            embarked_map = {"C": 0, "Q": 1, "S": 2}
            embarked = embarked_map[embarked_var.get()]
            
            # Calculate family size
            family_size = sibsp + parch + 1
            is_alone = 1 if family_size == 1 else 0
            
            # Create feature array WITH PROPER FEATURE NAMES
            # Method 1: Create a DataFrame with proper column names
            passenger_data = pd.DataFrame({
                'Pclass': [pclass],
                'Sex': [gender],
                'Age': [age],
                'SibSp': [sibsp],
                'Parch': [parch],
                'Fare': [fare],
                'Embarked': [embarked],
                'FamilySize': [family_size],
                'IsAlone': [is_alone]
            })
            
            # Make prediction using DataFrame (has feature names)
            prediction = best_model.predict(passenger_data)[0]
            probability = best_model.predict_proba(passenger_data)[0][1]
            
            # Display result
            if prediction == 1:
                result_text = "✅ SURVIVED!"
                color = "green"
                details = f"Survival Probability: {probability*100:.1f}%"
            else:
                result_text = "❌ DID NOT SURVIVE"
                color = "red"
                details = f"Survival Probability: {probability*100:.1f}%"
            
            result_label.config(text=f"{result_text}\n{details}", fg=color)
            
            # Show explanation
            explanation = get_prediction_explanation(pclass, gender, age, sibsp, parch, fare, family_size)
            messagebox.showinfo("Prediction Analysis", 
                              f"{result_text}\n\n{details}\n\n"
                              f"Key factors considered:\n{explanation}")
            
        except ValueError as e:
            messagebox.showerror("Input Error", 
                               "Please enter valid values for all fields!\n"
                               "• Age and Fare must be numbers\n"
                               "• Other fields should be whole numbers")
    
    def get_prediction_explanation(pclass, gender, age, sibsp, parch, fare, family_size):
        """Generate explanation for the prediction"""
        explanations = []
        
        # Gender analysis
        if gender == 1:  # female
            explanations.append("• Female passengers had 74% survival rate (vs 19% for males)")
        else:
            explanations.append("• Male passengers had 19% survival rate (vs 74% for females)")
        
        # Class analysis
        if pclass == 1:
            explanations.append("• 1st class passengers had 63% survival rate")
        elif pclass == 2:
            explanations.append("• 2nd class passengers had 47% survival rate")
        else:
            explanations.append("• 3rd class passengers had 24% survival rate")
        
        # Age analysis
        if age < 18:
            explanations.append("• Children (<18) had better survival chances (61%)")
        elif age > 60:
            explanations.append("• Elderly passengers had lower survival chances")
        else:
            explanations.append("• Adult passengers had average survival chances")
        
        # Family analysis
        if family_size == 1:
            explanations.append("• Traveling alone reduced survival chances (30%)")
        else:
            explanations.append("• Traveling with family improved survival chances (51%)")
        
        # Fare analysis (approximate)
        if fare > 100:
            explanations.append("• Higher fare indicates wealthier passenger (better survival)")
        elif fare < 20:
            explanations.append("• Lower fare indicates poorer passenger (worse survival)")
        
        return "\n".join(explanations)
    
    # Reset function
    def reset_inputs():
        pclass_var.set("1")
        gender_var.set("female")
        age_var.set("30")
        sibsp_var.set("0")
        parch_var.set("0")
        fare_var.set("50")
        embarked_var.set("S")
        result_label.config(text="Enter details and click PREDICT", fg='gray')
    
    # Buttons frame
    button_frame = tk.Frame(window, bg='#f0f0f0')
    button_frame.pack(pady=20)
    
    # Predict button
    predict_btn = tk.Button(button_frame, text="PREDICT SURVIVAL", 
                           command=predict_survival,
                           font=('Arial', 12, 'bold'),
                           bg='#3498db', fg='white',
                           padx=20, pady=10)
    predict_btn.pack(side='left', padx=10)
    
    # Reset button
    reset_btn = tk.Button(button_frame, text="RESET", 
                         command=reset_inputs,
                         font=('Arial', 12),
                         bg='#95a5a6', fg='white',
                         padx=20, pady=10)
    reset_btn.pack(side='left', padx=10)
    
    # Info button
    def show_model_info():
        info = f"""HOW THE PREDICTION WORKS:

Model: {best_name}
Accuracy: {best_acc*100:.1f}%

The model analyzes 9 features:
1. Passenger Class (1st/2nd/3rd)
2. Gender (Male/Female)
3. Age
4. Siblings/Spouses aboard
5. Parents/Children aboard
6. Ticket Fare
7. Port of Embarkation
8. Total Family Size
9. Traveling Alone or Not

Historical Patterns Learned:
• Females: 74% survived vs Males: 19%
• 1st Class: 63%, 2nd: 47%, 3rd: 24%
• Children (<10): 61% vs Adults: 38%
• With Family: 51% vs Alone: 30%

The model finds patterns in 891 real passenger records
to predict if a new passenger would survive."""
        messagebox.showinfo("Model Information", info)
    
    info_btn = tk.Button(button_frame, text="HOW IT WORKS", 
                        command=show_model_info,
                        font=('Arial', 12),
                        bg='#2ecc71', fg='white',
                        padx=20, pady=10)
    info_btn.pack(side='left', padx=10)
    
    # Footer
    footer = tk.Label(window, text="Based on Kaggle Titanic Dataset | Machine Learning Project", 
                     font=('Arial', 9), bg='#f0f0f0', fg='#7f8c8d')
    footer.pack(pady=20)
    
    # Start GUI
    window.mainloop()

# ====================== 8. EVALUATION ======================

print(f"\n📋 Classification Report for {best_name}:")
y_pred_best = y_pred_rf if best_name == "Random Forest" else y_pred_lr
print(classification_report(y_test, y_pred_best, target_names=['Not Survived', 'Survived']))

# Feature importance for Random Forest
if best_name == "Random Forest":
    print(f"\n🎯 Top 5 Feature Importance:")
    feat_imp = pd.DataFrame({
        'Feature': features,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    for idx, row in feat_imp.head(5).iterrows():
        print(f"   {row['Feature']}: {row['Importance']:.3f}")

print(f"\n🔮 Sample Predictions (Console):")
print("-"*40)

# Create sample passengers WITH PROPER FEATURE NAMES
samples = pd.DataFrame({
    'Pclass': [1, 3, 2],
    'Sex': [1, 0, 1],  # 1=female, 0=male
    'Age': [25, 30, 5],
    'SibSp': [0, 1, 2],
    'Parch': [0, 0, 2],
    'Fare': [50, 10, 25],
    'Embarked': [0, 1, 0],  # C=0, Q=1, S=2
    'FamilySize': [1, 2, 5],
    'IsAlone': [1, 0, 0]
})

predictions = best_model.predict(samples)
probabilities = best_model.predict_proba(samples)

for i in range(3):
    status = "SURVIVED ✅" if predictions[i] == 1 else "NOT SURVIVED ❌"
    prob = probabilities[i][1] * 100
    print(f"Passenger {i+1}: {status}")
    print(f"   Survival probability: {prob:.1f}%")
    print(f"   Features: Class={samples.iloc[i]['Pclass']}, Gender={'Female' if samples.iloc[i]['Sex']==1 else 'Male'}, Age={samples.iloc[i]['Age']}")
    print()

# ====================== 9. VISUALIZATION ======================
print(f"\n📈 Creating visualization...")

plt.figure(figsize=(10, 8))

# Plot 1: Survival by gender
plt.subplot(2, 2, 1)
survival_gender = df.groupby('Sex')['Survived'].mean()
plt.bar(['Male', 'Female'], survival_gender.values, color=['blue', 'pink'])
plt.title('Survival Rate by Gender')
plt.ylabel('Survival Rate')
for i, v in enumerate(survival_gender.values):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center')

# Plot 2: Survival by class
plt.subplot(2, 2, 2)
survival_class = df.groupby('Pclass')['Survived'].mean()
plt.bar(['1st', '2nd', '3rd'], survival_class.values, color=['gold', 'silver', 'brown'])
plt.title('Survival Rate by Passenger Class')
plt.ylabel('Survival Rate')
for i, v in enumerate(survival_class.values):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center')

# Plot 3: Age distribution of survivors vs non-survivors
plt.subplot(2, 2, 3)
survived_age = df[df['Survived'] == 1]['Age']
not_survived_age = df[df['Survived'] == 0]['Age']
plt.hist([survived_age, not_survived_age], bins=20, label=['Survived', 'Not Survived'], 
         color=['green', 'red'], alpha=0.7)
plt.title('Age Distribution by Survival')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend()

# Plot 4: Model comparison
plt.subplot(2, 2, 4)
models = ['Logistic Regression', 'Random Forest']
accuracies = [acc_lr, acc_rf]
bars = plt.bar(models, accuracies, color=['lightblue', 'lightgreen'])
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, acc + 0.02, 
             f'{acc:.3f}', ha='center')

plt.tight_layout()
plt.savefig('titanic_results.png', dpi=100, bbox_inches='tight')
print("✓ Visualization saved as 'titanic_results.png'")

# ====================== 10. SUMMARY ======================
print(f"\n" + "="*60)
print("PROJECT SUMMARY")
print("="*60)

# Calculate key statistics
gender_survival = df.groupby('Sex')['Survived'].mean()
class_survival = df.groupby('Pclass')['Survived'].mean()

summary = f"""
🎯 PROJECT: Titanic Survival Prediction
📁 DATASET: {len(df)} passengers, {len(features)} features
⚙️  MODELS: Logistic Regression & Random Forest
🏆 BEST MODEL: {best_name} ({best_acc*100:.1f}% accuracy)

🔑 KEY INSIGHTS:
1. Gender Survival Rates:
   • Female: {gender_survival[1]*100:.1f}% survived
   • Male: {gender_survival[0]*100:.1f}% survived

2. Class Survival Rates:
   • 1st Class: {class_survival[1]*100:.1f}% survived
   • 2nd Class: {class_survival[2]*100:.1f}% survived
   • 3rd Class: {class_survival[3]*100:.1f}% survived

3. Model Performance:
   • Random Forest: {acc_rf*100:.1f}%
   • Logistic Regression: {acc_lr*100:.1f}%

💾 OUTPUT:
   • titanic_results.png - All visualizations in one image

✅ PROJECT COMPLETED SUCCESSFULLY!
"""
print(summary)

# ====================== 11. LAUNCH GUI ======================
print("\n🖥️  Launching GUI window...")
print("Note: Console output is complete. GUI window will open separately.")

# Create and run GUI
create_gui()

TITANIC SURVIVAL PREDICTION PROJECT
✓ Loaded Kaggle Titanic dataset
   Shape: (891, 12)
   Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

📊 Basic Information:
   Total passengers: 891
   Survived: 342 (38.4%)
   Not Survived: 549

🔍 Missing Values:
   Age: 177 missing
   Cabin: 687 missing
   Embarked: 2 missing

⚙️  Data Preprocessing...
✓ Data split: Train=712, Test=179

🤖 Training Models...

🏆 RESULTS:
   Logistic Regression: 0.810 (81.0%)
   Random Forest: 0.816 (81.6%)

✓ Best Model: Random Forest with 81.6% accuracy

🖥️  Creating GUI...

📋 Classification Report for Random Forest:
              precision    recall  f1-score   support

Not Survived       0.79      0.95      0.86       110
    Survived       0.88      0.61      0.72        69

    accuracy                           0.82       179
   macro avg       0.83      0.78      0.79       179
weighted avg       0.83      0.82      0.81       179

